# Project 2 — Notebook 05: Visualizations & Business Insights

**Week 4 goal:** turn the retention and CLTV results into clear charts and recommendations.

## Step 1 — Import and load data

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_prep import (
    load_cohort_dataset,
    build_cohort_count_matrix,
    build_retention_matrix,
    add_customer_segments,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
df = load_cohort_dataset(DATA_DIR)
cohort_matrix = build_cohort_count_matrix(df)
retention_matrix = build_retention_matrix(cohort_matrix)
retention_percent = retention_matrix * 100
segmented = add_customer_segments(df)
print('Data ready')

## Chart 1 — Cohort retention heatmap

In [ ]:
plt.figure(figsize=(13, 7))
sns.heatmap(retention_percent, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Retention %'})
plt.title('Monthly Customer Retention by Cohort')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort Month')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cohort_retention_heatmap.png', dpi=150)
plt.show()

## Chart 2 — Retention decay curves for selected cohorts

In [ ]:
selected_cohorts = retention_percent.index[:6]

plt.figure(figsize=(11, 6))
for cohort in selected_cohorts:
    plt.plot(retention_percent.columns, retention_percent.loc[cohort], marker='o', label=str(cohort))

plt.title('Retention Curves for Early Cohorts')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Retention %')
plt.xticks(retention_percent.columns)
plt.ylim(0, 105)
plt.legend(title='Cohort')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'retention_curves.png', dpi=150)
plt.show()

## Chart 3 — Historical CLTV by cohort

In [ ]:
cohort_cltv = (df.groupby('cohort_month')
               .agg(customers=('customer_id', 'nunique'), revenue=('line_total', 'sum'))
               .reset_index())
cohort_cltv['historical_cltv'] = cohort_cltv['revenue'] / cohort_cltv['customers']
cohort_cltv['cohort_month'] = cohort_cltv['cohort_month'].astype(str)

plt.figure(figsize=(12, 6))
sns.barplot(data=cohort_cltv, x='cohort_month', y='historical_cltv', color='#4C78A8')
plt.title('Historical CLTV by Acquisition Cohort')
plt.xlabel('Cohort Month')
plt.ylabel('Historical CLTV (£ per customer)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cohort_cltv_bar_chart.png', dpi=150)
plt.show()

## Chart 4 — Revenue share by customer value segment

In [ ]:
segment_summary = (segmented.groupby('value_segment')
                   .agg(customers=('customer_id', 'nunique'), revenue=('line_total', 'sum'))
                   .reset_index())
segment_summary['revenue_share_percent'] = segment_summary['revenue'] / segment_summary['revenue'].sum() * 100

order = ['Low value', 'Medium value', 'High value']
plt.figure(figsize=(9, 6))
sns.barplot(data=segment_summary, x='value_segment', y='revenue_share_percent', order=order, palette='Blues')
plt.title('Revenue Share by Customer Value Segment')
plt.xlabel('Customer Segment')
plt.ylabel('Revenue Share %')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'segment_revenue_share.png', dpi=150)
plt.show()

segment_summary

## Chart 5 — Top countries by revenue

In [ ]:
country_revenue = (df.groupby('country')['line_total']
                   .sum()
                   .sort_values(ascending=False)
                   .head(10)
                   .reset_index())

plt.figure(figsize=(11, 6))
sns.barplot(data=country_revenue, y='country', x='line_total', color='#72B7B2')
plt.title('Top 10 Countries by Revenue')
plt.xlabel('Revenue (£)')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'top_countries_revenue.png', dpi=150)
plt.show()

## Business insights and recommendations

### Main findings
1. **Month 1 retention is low.** On average, only about 21% of customers come back the month after their first purchase.
2. **The December 2010 cohort looks strongest**, but it also contains customers already active at the start of the dataset, so I should be careful not to over-read it.
3. **Revenue is concentrated in the high-value segment.** The top 20% of customers bring in most of the revenue.
4. **The UK brings the most revenue by far**, while a few countries have high CLTV because they have fewer but larger customers/orders.

### Recommendations
- Send a re-engagement email or discount around 2–3 weeks after the first purchase, before Month 1 drop-off happens.
- Create a simple loyalty offer for repeat buyers, especially customers moving from medium value to high value.
- Track cohorts monthly going forward, not only once. Retention should become a regular business health metric.
- For finance, keep customer acquisition cost below the expected CLTV for each segment. Do not spend the same CAC for every customer group.

These are realistic first recommendations based on the data, not final answers. More data about marketing channels or product categories would make the recommendations stronger.